# **Notebook Penelitian Skripsi: Deteksi & Segmentasi Lubang Jalan Raya**
Notebook ini didesain khusus untuk kebutuhan akademis. Seluruh proses (Preprocessing, Augmentasi, dll) dilakukan pada **seluruh dataset**, bukan sampel. Parameter dan hasil setiap tahapan ditampilkan dalam bentuk **Tabel (Dataframe)** untuk mempermudah penyusunan skripsi.

---

## **1. Mengimpor Pustaka (Import Libraries)**

In [ ]:
import os
import cv2
import glob
import shutil
import gc
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Instalasi dependensi utama
!pip install -q ultralytics albumentations roboflow
from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG
import albumentations as A

print("Pustaka berhasil diimpor.")
print(f"CUDA/GPU Tersedia: {torch.cuda.is_available()}")

## **2. Pengumpulan Dataset (Data Preparation)**
Mengunduh dataset dari Roboflow dan menghitung total data valid/tidak valid di setiap pembagian (train, valid, test), lalu menampilkannya sebagai tabel.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="PWjvZLsn6rqdrPEFBpss")
project = rf.workspace("juan-wens").project("pothole-detection-ml2lx-jrazm")
version = project.version(1)
dataset = version.download("yolov9")

dataset_dir = dataset.location
splits = ['train', 'valid', 'test']
data_stats = []

# Membaca seluruh file di dataset
for split in splits:
    img_paths = glob.glob(os.path.join(dataset_dir, split, "images", "*.*"))
    valid_count = 0
    invalid_count = 0
    
    for img_path in img_paths:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(dataset_dir, split, "labels", base_name + ".txt")
        
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            valid_count += 1
        else:
            invalid_count += 1
            
    data_stats.append({
        "Fase": split.upper(),
        "Total Gambar": len(img_paths),
        "Data Valid (Ada Lubang)": valid_count,
        "Data Tidak Valid (Jalan Mulus)": invalid_count
    })

df_stats = pd.DataFrame(data_stats)
display(df_stats)

if os.path.exists("/content/dataset"):
    if os.path.islink("/content/dataset"):
        os.unlink("/content/dataset")
    else:
        shutil.rmtree("/content/dataset")
os.symlink(dataset_dir, "/content/dataset")

## **3. Pra-pemrosesan Data (Pre-processing Data)**
Tahap ini secara fisik memodifikasi **seluruh gambar** dalam dataset menjadi ukuran 640x640. Parameter dan status eksekusi disajikan dalam tabel.

In [ ]:
prep_results = []
target_size = (640, 640)

for split in splits:
    img_paths = glob.glob(os.path.join("/content/dataset", split, "images", "*.*"))
    count = 0
    for img_path in img_paths:
        img = cv2.imread(img_path)
        if img is not None:
            resized = cv2.resize(img, target_size)
            cv2.imwrite(img_path, resized)
            del img, resized # Membersihkan memori gambar tiap iterasi agar RAM tidak menumpuk
            count += 1
            
    prep_results.append({
        "Folder Dataset": split.upper(),
        "Tahap Pre-processing": "Resize & Normalization",
        "Resolusi Target": f"{target_size[0]}x{target_size[1]}",
        "Jumlah Gambar Diproses": count,
        "Status": "Selesai"
    })

gc.collect() # Memaksa pembersihan RAM setelah preprocessing
df_prep = pd.DataFrame(prep_results)
display(df_prep)

## **4. Augmentasi Data (Data Augmentation)**
Pada arsitektur YOLOv9, augmentasi data dilakukan secara otomatis dan dinamis (on-the-fly) selama proses pelatihan berlangsung, sehingga tidak perlu merusak file gambar asli. Tabel berikut menampilkan hyperparameter augmentasi yang diekstrak langsung dari konfigurasi bawaan YOLO.

In [ ]:
# Mengambil parameter augmentasi langsung dari konfigurasi internal YOLO
aug_keys = ['hsv_h', 'hsv_s', 'hsv_v', 'degrees', 'translate', 'scale', 'shear', 'perspective', 'flipud', 'fliplr', 'mosaic', 'mixup', 'copy_paste']
aug_results = []

for key in aug_keys:
    aug_results.append({
        "Teknik Augmentasi (Internal YOLO)": key,
        "Probabilitas / Batas Nilai": getattr(DEFAULT_CFG, key),
        "Keterangan": "Diterapkan otomatis saat epoch berjalan"
    })

df_aug = pd.DataFrame(aug_results)
display(df_aug)

## **5. Ekstraksi Fitur dan Arsitektur Model (Model Architecture)**
Di sini, program memuat model YOLOv9 pra-latih (Pre-trained) dan secara dinamis membaca tipe lapisan (layer) arsitekturnya langsung dari objek PyTorch.

In [ ]:
model = YOLO('yolov9c.pt', task='detect')

arch_info = []

# Mengekstrak informasi layer arsitektur murni dari internal model Pytorch
for i, layer in enumerate(model.model.model):
    # Hitung bobot parameter per layer
    params = sum(p.numel() for p in layer.parameters())
    arch_info.append({
        "Layer Index": i,
        "Tipe Arsitektur Layer": type(layer).__name__,
        "Jumlah Parameter": f"{params:,}",
        "Sumber": "Diekstrak langsung dari pt file"
    })

df_arch = pd.DataFrame(arch_info)
# Menampilkan cuplikan 10 layer pertama arsitektur YOLOv9 (biasanya Backbone)
display(df_arch.head(10))

total_params = sum(p.numel() for p in model.model.parameters())
print(f"\nTotal Keseluruhan Parameter Model: {total_params / 1e6:.2f} Juta (M)")

## **6. Kompilasi Model (Model Compilation)**
Pada implementasi YOLO, proses kompilasi direpresentasikan oleh definisi Hyperparameter dan fungsi Loss. Data tabel berikut ini murni diekstrak dari konfigurasi internal YOLO (`DEFAULT_CFG`).

In [ ]:
compilation_params = []
# Key dari parameter optimizer, learning rate, dan loss function milik YOLO
comp_keys = ['optimizer', 'lr0', 'lrf', 'momentum', 'weight_decay', 'box', 'cls', 'dfl']

for key in comp_keys:
    compilation_params.append({
        "Parameter Kompilasi": key,
        "Nilai (Diekstrak dari Model)": getattr(DEFAULT_CFG, key),
        "Sumber": "ultralytics DEFAULT_CFG"
    })

df_compile = pd.DataFrame(compilation_params)
display(df_compile)

## **7. Pelatihan Model (Model Training)**
Pembuatan konfigurasi `data.yaml` dan eksekusi pelatihan. Tabel di bawah dibangun langsung dari *dictionary argument*, yang kemudian parameter spesifiknya dilempar ke fungsi `.train()`.

In [ ]:
import time
yaml_content = f"""
train: /content/dataset/train/images
val: /content/dataset/valid/images
test: /content/dataset/test/images
nc: 1
names: ['pothole']
"""
with open("/content/dataset/data.yaml", "w") as f:
    f.write(yaml_content.strip())

# Hapus cache lama jika ada agar YOLO tidak kebingungan dengan memori dari sesi error sebelumnya
for cache_file in glob.glob('/content/dataset/**/labels.cache', recursive=True):
    try: os.remove(cache_file)
    except: pass
for cache_file in glob.glob('/content/Pothole*/**/labels.cache', recursive=True):
    try: os.remove(cache_file)
    except: pass

# Menyimpan argumen ke variabel untuk dokumentasi tabel yang dinamis (tidak statis)
train_args = {
    "data": "/content/dataset/data.yaml",
    "epochs": 30, # [CPU OPTIMASI] Diturunkan ke 30 agar lebih cepat selesai di CPU.
    "imgsz": 320, # [CPU OPTIMASI] Diturunkan ke 320. Resolusi diperkecil agar beban komputasi CPU berkurang drastis (sangat signifikan).
    "batch": 4,   # Wajib 4 agar memori RAM Colab gratisan tidak crash.
    "device": 0 if torch.cuda.is_available() else "cpu",
    "optimizer": "auto",
    "project": "skripsi_deteksi_lubang",
    "name": "yolov9_pothole_det",
    "workers": 0, # Wajib 0 untuk CPU agar tidak crash multiprocessing.
    "cache": False # Wajib False agar RAM Colab gratisan tidak over-limit.
}

df_train = pd.DataFrame(list(train_args.items()), columns=["Parameter Training", "Nilai yang Ditetapkan"])
display(df_train)

# --- MEMAKSA RE-INISIALISASI MODEL ---
# Langkah ini menjamin memori Jupyter Notebook memakai model DETEKSI murni
# meskipun Anda belum melakukan 'Restart Runtime' pada Colab.
model = YOLO('yolov9c.pt', task='detect')
gc.collect() # Bersihkan RAM semaksimal mungkin sebelum beban berat

print("\n--- MEMULAI PELATIHAN MODEL ---")
start_time = time.time() # Memulai perhitungan stopwatch

# Mengeksekusi pelatihan secara eksplisit tanpa menggunakan unpack dictionary (**kwargs)
# untuk menghindari error "'dict' object is not callable" pada versi Python tertentu.
results = model.train(
    task='detect',
    data=train_args["data"],
    epochs=train_args["epochs"],
    imgsz=train_args["imgsz"],
    batch=train_args["batch"],
    device=train_args["device"],
    optimizer=train_args["optimizer"],
    project=train_args["project"],
    name=train_args["name"],
    workers=train_args["workers"],
    cache=train_args["cache"]
)

end_time = time.time() # Menghentikan perhitungan stopwatch
training_duration = end_time - start_time


## **7.5. Menghitung Waktu Pelatihan (Training Time)**
Sel ini mengonversi total durasi waktu yang dihabiskan model untuk berlatih dari format detik menjadi format Jam:Menit:Detik yang mudah dibaca, serta menampilkannya dalam bentuk tabel.

In [ ]:
if 'training_duration' in locals():
    hours, rem = divmod(training_duration, 3600)
    minutes, seconds = divmod(rem, 60)
    waktu_teks = f"{int(hours):02d} Jam, {int(minutes):02d} Menit, {seconds:.2f} Detik"
    
    df_waktu = pd.DataFrame([{
        "Total Detik": f"{training_duration:.2f} s",
        "Durasi Terbaca (Human Readable)": waktu_teks
    }])
    
    print("\n======================================================")
    print(" WAKTU TOTAL PELATIHAN MODEL")
    print("======================================================")
    display(df_waktu)
else:
    print("Variabel 'training_duration' belum ada. Pastikan Anda telah menjalankan Tahap 7 secara utuh sampai selesai.")

## **8. Evaluasi Model dan Confusion Matrix (Model Evaluation)**

Tahap ini membahas hasil performa model setelah dilatih. Menggunakan matplotlib untuk memvisualisasikan data murni dari `results.csv`.

### **A. Pembahasan Metrik Evaluasi**
1. **Precision (Presisi):** Mengukur ketepatan model. Jika model mengatakan itu lubang, seberapa yakin itu benar-benar lubang? (Fokus meminimalkan *False Positive* atau salah menuduh jalan mulus sebagai lubang).
2. **Recall (Sensitivitas):** Mengukur kemampuan temuan. Dari seluruh lubang aktual yang ada di aspal, berapa persen yang berhasil ditemukan oleh model? (Fokus meminimalkan *False Negative* atau lubang yang terlewat).
3. **mAP50 (Box):** *Mean Average Precision* pada threshold kecocokan 50%. Ini adalah skor akurasi utama yang menunjukkan performa kotak pembatas (Box).
4. **mAP50-95:** Metrik sangat ketat yang merata-ratakan akurasi di berbagai tingkat kecocokan (50% hingga 95%). Menunjukkan seberapa akurat posisi kotak pembatas model mengikuti tepian lubang.

### **B. Pembahasan Confusion Matrix**
Confusion Matrix memetakan prediksi model melawan kenyataan (Ground Truth):
* **True Positive (Kiri Atas):** Pothole (lubang) ditebak sebagai Pothole (Benar).
* **False Negative (Bawah Kiri):** Pothole (lubang) ditebak sebagai Background/Jalan Mulus (Terlewat / Gagal deteksi).
* **False Positive (Kanan Atas):** Background (jalan mulus) ditebak sebagai Pothole (Salah tebak / *Ghost detection*).
* **True Negative (Kanan Bawah):** Background dibiarkan sebagai Background (Benar).

In [ ]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# PERBAIKAN TOTAL: Kita gunakan pencarian rekursif di seluruh direktori /content/
# Hal ini menjamin file results.csv PASTI KETEMU meskipun strukturnya diacak Colab.
all_csv_paths = glob.glob('/content/**/yolov9_pothole_det*/results.csv', recursive=True)

if not all_csv_paths:
    print("❌ ERROR: File results.csv sama sekali tidak ditemukan di Google Colab Anda.")
    print("Kemungkinan besar pelatihan di Tahap 7 belum mencapai Epoch pertama, atau terhenti di tengah jalan.")
else:
    # Ambil file CSV yang dimodifikasi paling terakhir (terbaru)
    latest_csv = max(all_csv_paths, key=os.path.getmtime)
    result_path = os.path.dirname(latest_csv)
    
    print(f"✅ Membaca data metrik dari folder terbaru: {result_path}\n")
    
    df_res = pd.read_csv(latest_csv)
    # Mengambil nama kolom dan membersihkan spasi
    df_res.columns = df_res.columns.str.strip()
    
    print("======================================================")
    print(" 1. METRICS SUMMARY (Diambil dari Epoch Terakhir)")
    print("======================================================")
    # Ambil baris terakhir (epoch terakhir)
    latest = df_res.iloc[-1]
    p = latest['metrics/precision(B)']
    r = latest['metrics/recall(B)']
    f1_score = (2 * p * r) / (p + r) if (p + r) > 0 else 0
    
    # Format ke dalam DataFrame untuk tampilan persis seperti dashboard
    df_metrics = pd.DataFrame([{
        'mAP@50': f"{latest['metrics/mAP50(B)']*100:.1f}%",
        'Precision': f"{p*100:.1f}%",
        'Recall': f"{r*100:.1f}%",
        'F1 Score (External)': f"{f1_score*100:.1f}%"
    }])
    display(df_metrics)
    
    print("\n======================================================")
    print(" 2. GRAFIK MODEL PERFORMANCE (mAP over Epochs)")
    print("======================================================")
    plt.figure(figsize=(10, 5))
    # Plot mAP50 dan mAP50-95 mirip grafik Roboflow/W&B
    plt.plot(df_res['epoch'], df_res['metrics/mAP50(B)']*100, label='mAP', color='blueviolet', linewidth=2)
    plt.plot(df_res['epoch'], df_res['metrics/mAP50-95(B)']*100, label='mAP@50:95', color='lavender', linewidth=2)
    plt.title('Model Performance', loc='left', pad=15, fontweight='bold', color='dimgray')
    plt.xlabel('Epochs')
    # Mengatur limit Y ke 0-100%
    plt.ylim(0, 100)
    # Formatter sumbu Y ke persen
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:.1f}%".format(x)))
    plt.grid(True, linestyle='-', alpha=0.3)
    plt.legend(loc='upper right')
    # Menghilangkan border frame atas dan kanan
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.show()
    
    print("\n======================================================")
    print(" 3. GRAFIK TRAINING LOSSES (Box, Class, Object/DFL)")
    print("======================================================")
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    
    # Cek nama kolom loss (tergantung versi YOLO, bisa train/box_loss atau val/box_loss)
    # Kita gunakan train/ loss untuk menampilkan grafik kerugian saat epoch berlangsung
    loss_cols = ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']
    titles = ['Box Loss', 'Class Loss', 'Object (DFL) Loss']
    
    for i, (col, title) in enumerate(zip(loss_cols, titles)):
        if col in df_res.columns:
            axs[i].plot(df_res['epoch'], df_res[col], color='blueviolet', label=title)
            axs[i].set_title(title, loc='left', fontweight='bold', color='dimgray')
            axs[i].set_xlabel('Epochs')
            axs[i].grid(True, linestyle='-', alpha=0.3)
            axs[i].legend(loc='upper right')
            axs[i].spines['top'].set_visible(False)
            axs[i].spines['right'].set_visible(False)
            
    plt.tight_layout()
    plt.show()

    cm_path = os.path.join(result_path, 'confusion_matrix.png')
    if os.path.exists(cm_path):
        print("\n======================================================")
        print(" 4. VISUALISASI CONFUSION MATRIX")
        print("======================================================")
        img_cm = cv2.imread(cm_path)
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(img_cm, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()
    else:
        print("⚠️ Info: Gambar Confusion Matrix belum dihasilkan oleh YOLO.")


## **9. Penyimpanan Model (Model Saving)**
Menyimpan arsitektur dan bobot hasil pembelajaran dalam format `.pt`.

In [ ]:
from google.colab import files
import glob
import os

# Sama seperti di atas, kita gunakan pencarian rekursif
all_weights = glob.glob('/content/**/yolov9_pothole_det*/weights/best.pt', recursive=True)

if all_weights:
    best_weight = max(all_weights, key=os.path.getmtime)
    print(f"✅ File model terbaik ditemukan di: {best_weight}")
    print("Sedang mengunduh file model...")
    files.download(best_weight)
else:
    print("❌ Belum ada file best.pt yang ditemukan. Pastikan training selesai.")